## 1. Importar pandas y definir la ruta del archivo

In [1]:
import pandas as pd

df = pd.read_csv('Notas_de_estudiantes.csv')
df.head()

,nombre,apellido,semestre,edad,nota
0,Natalia,Mendoza,1ro,25,"6,09"
1,pedro,A guilar,1,24,"""2,51"""
2,Ramon,Rivera,-,25,"02,93"
3,Gabriela,Flore,8,85,"5,71"
4,Carlos,Martinez,3,,NaN


In [4]:
#Revision de datos
print("Resumen de estructura:")
df.info()

print("\nTipos de datos:")
display(df.dtypes)

print("\nValores nulos por columna:")
display(df.isna().sum())



Resumen de estructura:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   nombre    191 non-null    object
 1   apellido  196 non-null    object
 2   semestre  184 non-null    object
 3   edad      190 non-null    object
 4   nota      192 non-null    object
dtypes: object(5)
memory usage: 7.9+ KB

Tipos de datos:


nombre      object
apellido    object
semestre    object
edad        object
nota        object
dtype: object


Valores nulos por columna:


nombre       9
apellido     4
semestre    16
edad        10
nota         8
dtype: int64

## 2. Identificación de problemas de calidad
a) Valores faltantes
b) Duplicados
c) Outliers en `nota` (< 1.0 o > 7.0)

In [6]:
# Copia para evaluación de calidad
df_eval = df.copy()

# Normalizar texto y marcar tokens comunes de faltantes
for col in df_eval.columns:
    df_eval[col] = df_eval[col].astype("string").str.strip()

tokens_faltantes = {"", "na", "n/a", "null", "none"}
for col in df_eval.columns:
    lower_col = df_eval[col].str.lower()
    df_eval[col] = df_eval[col].where(~lower_col.isin(tokens_faltantes), pd.NA)

# a) Valores faltantes
faltantes_por_columna = df_eval.isna().sum()
total_faltantes = int(faltantes_por_columna.sum())

print("a) Valores faltantes")
print("Total de valores faltantes:", total_faltantes)
display(faltantes_por_columna)

# b) Duplicados
duplicados_filas = int(df_eval.duplicated().sum())
duplicados_estudiante = df_eval.duplicated(subset=["nombre", "apellido"], keep=False)
cant_duplicados_estudiante = int(duplicados_estudiante.sum())

print("\nb) Duplicados")
print("Filas totalmente duplicadas:", duplicados_filas)
print("Registros con nombre+apellido repetido:", cant_duplicados_estudiante)
if cant_duplicados_estudiante > 0:
    display(df_eval.loc[duplicados_estudiante, ["nombre", "apellido", "semestre", "edad", "nota"]].sort_values(["nombre", "apellido"]))

# c) Outliers en nota (<1.0 o >7.0)
nota_limpia = (
    df_eval["nota"]
    .str.replace('"', "", regex=False)
    .str.replace("pts", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.extract(r"([0-9]*\.?[0-9]+)", expand=False)
)
df_eval["nota_num"] = pd.to_numeric(nota_limpia, errors="coerce")

outliers_nota = df_eval[(df_eval["nota_num"] < 1.0) | (df_eval["nota_num"] > 7.0)]

print("\nc) Outliers en nota (<1.0 o >7.0)")
print("Cantidad de outliers:", len(outliers_nota))
if len(outliers_nota) > 0:
    display(outliers_nota[["nombre", "apellido", "nota", "nota_num"]].sort_values("nota_num"))

a) Valores faltantes
Total de valores faltantes: 67


nombre      16
apellido     9
semestre    17
edad        15
nota        10
dtype: int64


b) Duplicados
Filas totalmente duplicadas: 25
Registros con nombre+apellido repetido: 64


,nombre,apellido,semestre,edad,nota
62,Andres,Vargas,7,21,"""9,17"""
77,Andres,Vargas,4,20,"7,34"
97,Beatriz,diaz,primero,26,999
187,Beatriz,diaz,primero,26,999
50,Camila,Perez,9,25,"5,99"
...,...,...,...,...,...
66,<NA>,morales,8,22,"8,44"
67,<NA>,morales,8,22,"8,44"
6,<NA>,<NA>,3,23,"""3,29 pts"""
63,<NA>,<NA>,2,-5,"$5,82"



c) Outliers en nota (<1.0 o >7.0)
Cantidad de outliers: 82


,nombre,apellido,nota,nota_num
181,Laura,Vargas,0,0.0
163,Claudia,Sanchez,"$0,1",0.1
7,Beatriz,Gomez,"0,41",0.41
132,Fernando,Paredes,"00,41",0.41
135,Fernando,Paredes,"00,41",0.41
...,...,...,...,...
97,Beatriz,diaz,999,999.0
24,<NA>,Gomez,999,999.0
187,Beatriz,diaz,999,999.0
23,Luis,Lopez,1000,1000.0


## 3. Aplicación de métodos de limpieza
Se aplican tres métodos:
1. Imputación de faltantes con mediana (columnas numéricas).
2. Eliminación de filas duplicadas.
3. Corrección de outliers en `nota` ajustando al rango [1.0, 7.0].

In [8]:
# Pipeline de limpieza aplicado
df_limpio = df.copy()

# 1) Normalización y faltantes explícitos
for col in df_limpio.columns:
    df_limpio[col] = df_limpio[col].astype("string").str.strip()

tokens_faltantes = {"", "na", "n/a", "null", "none"}
for col in df_limpio.columns:
    df_limpio[col] = df_limpio[col].mask(df_limpio[col].str.lower().isin(tokens_faltantes), pd.NA)

# Convertir columnas numéricas
nota_num = (
    df_limpio["nota"]
    .str.replace('"', "", regex=False)
    .str.replace("pts", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.extract(r"([0-9]*\.?[0-9]+)", expand=False)
)
df_limpio["nota_num"] = pd.to_numeric(nota_num, errors="coerce")
df_limpio["edad_num"] = pd.to_numeric(df_limpio["edad"], errors="coerce")

# Métricas antes de limpiar
faltantes_antes = int(df_limpio.isna().sum().sum())
duplicados_antes = int(df_limpio.duplicated().sum())
outliers_antes = int(((df_limpio["nota_num"] < 1.0) | (df_limpio["nota_num"] > 7.0)).sum())

# 2) Imputación de faltantes con mediana
df_limpio["edad_num"] = df_limpio["edad_num"].fillna(df_limpio["edad_num"].median())
df_limpio["nota_num"] = df_limpio["nota_num"].fillna(df_limpio["nota_num"].median())

# 3) Eliminar duplicados
df_limpio = df_limpio.drop_duplicates().copy()

# 4) Corregir outliers ajustando al rango [1, 7]
df_limpio["nota_num"] = df_limpio["nota_num"].clip(lower=1.0, upper=7.0)

# Métricas después de limpiar
faltantes_despues = int(df_limpio[["edad_num", "nota_num"]].isna().sum().sum())
duplicados_despues = int(df_limpio.duplicated().sum())
outliers_despues = int(((df_limpio["nota_num"] < 1.0) | (df_limpio["nota_num"] > 7.0)).sum())

print("Resumen de limpieza aplicada")
print("- Faltantes (edad_num + nota_num) antes:", faltantes_antes)
print("- Faltantes (edad_num + nota_num) después:", faltantes_despues)
print("- Duplicados antes:", duplicados_antes)
print("- Duplicados después:", duplicados_despues)
print("- Outliers de nota antes:", outliers_antes)
print("- Outliers de nota después:", outliers_despues)

display(df_limpio[["nombre", "apellido", "edad_num", "nota_num"]].head())

Resumen de limpieza aplicada
- Faltantes (edad_num + nota_num) antes: 108
- Faltantes (edad_num + nota_num) después: 0
- Duplicados antes: 25
- Duplicados después: 0
- Outliers de nota antes: 82
- Outliers de nota después: 0


,nombre,apellido,edad_num,nota_num
0,Natalia,Mendoza,25,6.09
1,pedro,A guilar,24,2.51
2,Ramon,Rivera,25,2.93
3,Gabriela,Flore,85,5.71
4,Carlos,Martinez,24,5.6


## 4. Comentario sobre decisiones de limpieza
**a) Valores faltantes**: se imputaron con la **mediana** en `edad_num` y `nota_num`.
Se eligió mediana porque es más robusta que la media frente a valores extremos (por ejemplo, notas como 999 o 1000) y reduce el sesgo en los datos.

**b) Duplicados**: se eliminaron filas duplicadas exactas con `drop_duplicates()`.
Se tomó esta decisión para evitar contar el mismo registro más de una vez y no distorsionar estadísticas como promedio, distribución de notas o cantidad de estudiantes.

**c) Outliers en nota**: se corrigieron ajustando los valores al rango válido **[1.0, 7.0]** usando `clip`.
Se prefirió ajustar en lugar de eliminar para conservar la mayor cantidad de registros posible, manteniendo el criterio académico esperado para la escala de notas.

In [2]:

import re

# ── Copia de trabajo ──────────────────────────────────────────────────────────
df_q = df.copy()

# Normalizar: quitar espacios y uniformar tokens de "vacío"
for col in df_q.columns:
    df_q[col] = df_q[col].astype(str).str.strip()

TOKENS_NULOS = {"", "na", "n/a", "null", "none", "nan", "-", "?", "nulo", "nd", "sin nota",
                "aprobado", "s/c", "nulnulo", "nulnull"}

def es_nulo(valor):
    return str(valor).strip().lower() in TOKENS_NULOS

# ── Helpers de validación por columna ────────────────────────────────────────

def nombre_erroneo(v):
    v = str(v).strip()
    if es_nulo(v): return True
    if re.search(r'\s', v): return True   # espacios dentro del nombre (ej. "S ofia")
    return False

def semestre_erroneo(v):
    v = str(v).strip().lower()
    if es_nulo(v) or v == "": return True
    # palabras como "primero", "octavo"
    if re.fullmatch(r'[a-záéíóú]+', v) and not re.fullmatch(r'(i{1,3}|iv|vi{0,3}|ix|x)', v): return True
    # romano válido (I–X) → aceptar
    if re.fullmatch(r'(i{1,3}|iv|vi{0,3}|ix|x)', v): return False
    # "1ro", "2do", etc.
    if re.fullmatch(r'\d+(ro|do|to|vo)', v): return True
    try:
        n = int(v)
        return n < 1 or n > 10
    except:
        return True

def edad_errónea(v):
    v = str(v).strip().lower()
    if es_nulo(v) or v == "": return True
    try:
        n = int(v)
        return n < 15 or n > 60
    except:
        return True

def nota_errónea(v):
    v = str(v).strip().strip('"').lower()
    if es_nulo(v) or v == "": return True
    # limpiar: quitar pts, /10, $, reemplazar coma
    v = re.sub(r'\s*pts\s*$', '', v)
    v = re.sub(r'/10$', '', v)
    v = re.sub(r'^\$', '', v)
    v = v.replace(',', '.').strip()
    try:
        n = float(v)
        return n < 1.0 or n > 7.0
    except:
        return True

# ── Aplicar validaciones ──────────────────────────────────────────────────────
df_q['err_nombre']   = df_q['nombre'].apply(nombre_erroneo)
df_q['err_apellido'] = df_q['apellido'].apply(nombre_erroneo)
df_q['err_semestre'] = df_q['semestre'].apply(semestre_erroneo)
df_q['err_edad']     = df_q['edad'].apply(edad_errónea)
df_q['err_nota']     = df_q['nota'].apply(nota_errónea)

err_cols = ['err_nombre', 'err_apellido', 'err_semestre', 'err_edad', 'err_nota']
data_cols = ['nombre', 'apellido', 'semestre', 'edad', 'nota']

total_filas  = len(df_q)
total_celdas = total_filas * len(data_cols)

# Errores por columna
errores_col = {col: int(df_q[ecol].sum()) for col, ecol in zip(data_cols, err_cols)}
total_errores = sum(errores_col.values())

# Filas con al menos un error
filas_con_error = int(df_q[err_cols].any(axis=1).sum())

# ── Resultados ────────────────────────────────────────────────────────────────
resumen = pd.DataFrame({
    'Columna': data_cols,
    'Celdas erróneas': [errores_col[c] for c in data_cols],
    '% sobre total filas': [round(errores_col[c] / total_filas * 100, 1) for c in data_cols]
})

print("=" * 50)
print(f"  Total de registros   : {total_filas}")
print(f"  Total de celdas      : {total_celdas}")
print(f"  Celdas con error     : {total_errores}")
print(f"  % celdas erróneas    : {total_errores / total_celdas * 100:.1f}%")
print(f"  Filas con ≥1 error   : {filas_con_error}")
print(f"  % filas con error    : {filas_con_error / total_filas * 100:.1f}%")
print("=" * 50)
display(resumen)


  Total de registros   : 200
  Total de celdas      : 1000
  Celdas con error     : 235
  % celdas erróneas    : 23.5%
  Filas con ≥1 error   : 163
  % filas con error    : 81.5%


,Columna,Celdas erróneas,% sobre total filas
0,nombre,28,14.0
1,apellido,23,11.5
2,semestre,40,20.0
3,edad,37,18.5
4,nota,107,53.5


In [6]:

import re
import pandas as pd

# ── Cargar desde el CSV original ──────────────────────────────────────────────
df_clean = pd.read_csv('Notas_de_estudiantes.csv')

# ── 1. Normalizar strings y marcar nulos ──────────────────────────────────────
NULOS = {"", "na", "n/a", "null", "none", "nan", "-", "?", "nulo", "nd",
         "sin nota", "aprobado", "s/c", "nnull", "nulnulo"}

for col in df_clean.columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()
    df_clean[col] = df_clean[col].where(
        ~df_clean[col].str.lower().isin(NULOS), other=pd.NA
    )

# ── 2. Limpiar NOMBRE y APELLIDO ──────────────────────────────────────────────
# Quitar espacios internos (ej. "S ofia" → "Sofia"), title case
for col in ['nombre', 'apellido']:
    df_clean[col] = df_clean[col].str.replace(r'\s+', '', regex=True).str.title()
    # Si quedó vacío tras limpiar, marcar como NA
    df_clean[col] = df_clean[col].where(df_clean[col].str.strip() != '', other=pd.NA)
    # Imputar con "Desconocido"
    df_clean[col] = df_clean[col].fillna('Desconocido')

# ── 3. Limpiar SEMESTRE ───────────────────────────────────────────────────────
SEMESTRE_PALABRAS = {
    'primero': 1, 'segundo': 2, 'tercero': 3, 'cuarto': 4,
    'quinto': 5, 'sexto': 6, 'séptimo': 7, 'septimo': 7,
    'octavo': 8, 'noveno': 9, 'décimo': 10, 'decimo': 10
}
ROMANOS = {'i': 1, 'ii': 2, 'iii': 3, 'iv': 4, 'v': 5,
           'vi': 6, 'vii': 7, 'viii': 8, 'ix': 9, 'x': 10}

def limpiar_semestre(v):
    if pd.isna(v): return pd.NA
    v = str(v).strip().lower()
    v = re.sub(r'\s+', '', v)      # quitar espacios internos
    if v in NULOS: return pd.NA
    if v in SEMESTRE_PALABRAS: return SEMESTRE_PALABRAS[v]
    if v in ROMANOS: return ROMANOS[v]
    # "1ro", "2do", etc.
    m = re.fullmatch(r'(\d+)(ro|do|to|vo|rd|th|st|nd)', v)
    if m: return int(m.group(1))
    try:
        n = int(v)
        return n if 1 <= n <= 10 else pd.NA
    except:
        return pd.NA

df_clean['semestre'] = df_clean['semestre'].apply(limpiar_semestre)
mediana_sem = int(round(pd.to_numeric(df_clean['semestre'], errors='coerce').median()))
df_clean['semestre'] = pd.to_numeric(df_clean['semestre'], errors='coerce').fillna(mediana_sem).astype(int)

# ── 4. Limpiar EDAD ───────────────────────────────────────────────────────────
df_clean['edad'] = (
    df_clean['edad']
    .str.replace(r'\s+', '', regex=True)
    .pipe(lambda s: pd.to_numeric(s, errors='coerce'))
)
# Edades fuera de rango [15, 60] → NaN
df_clean.loc[(df_clean['edad'] < 15) | (df_clean['edad'] > 60), 'edad'] = pd.NA
mediana_edad = df_clean['edad'].median()
df_clean['edad'] = df_clean['edad'].fillna(mediana_edad).round().astype(int)

# ── 5. Limpiar NOTA ───────────────────────────────────────────────────────────
def limpiar_nota(v):
    if pd.isna(v): return pd.NA
    v = str(v).strip().strip('"').lower()
    if v in NULOS: return pd.NA
    v = re.sub(r'\s*pts?\.?\s*$', '', v)
    v = re.sub(r'/10$', '', v)
    v = re.sub(r'^\$', '', v)
    v = v.replace(',', '.').strip()
    # quitar ceros iniciales extra: "02,93" → "2.93"
    try:
        return float(v)
    except:
        return pd.NA

df_clean['nota'] = df_clean['nota'].apply(limpiar_nota)
# Notas fuera de [1, 7] → clip
df_clean['nota'] = pd.to_numeric(df_clean['nota'], errors='coerce')
mediana_nota = df_clean.loc[(df_clean['nota'] >= 1) & (df_clean['nota'] <= 7), 'nota'].median()
df_clean['nota'] = df_clean['nota'].fillna(mediana_nota)
df_clean['nota'] = df_clean['nota'].clip(lower=1.0, upper=7.0).round(2)

# ── 6. Eliminar duplicados exactos ────────────────────────────────────────────
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# ── Verificación final ────────────────────────────────────────────────────────
print("✔ Dataset limpio — df.info():")
df_clean.info()
print(f"\nForma final: {df_clean.shape}")
df_clean.head(173)


✔ Dataset limpio — df.info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173 entries, 0 to 172
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   nombre    173 non-null    object 
 1   apellido  173 non-null    object 
 2   semestre  173 non-null    int64  
 3   edad      173 non-null    int64  
 4   nota      173 non-null    float64
dtypes: float64(1), int64(2), object(2)
memory usage: 6.9+ KB

Forma final: (173, 5)


,nombre,apellido,semestre,edad,nota
0,Natalia,Mendoza,1,25,6.09
1,Pedro,Aguilar,1,24,2.51
2,Ramon,Rivera,5,25,2.93
3,Gabriela,Flore,8,24,5.71
4,Carlos,Martinez,3,24,3.29
...,...,...,...,...,...
168,Beatriz,Morales,9,29,7.00
169,Natalia,Ruiz,1,29,2.54
170,Diego,Ramirez,2,24,3.29
171,Desconocido,Silva,4,24,7.00


## 5. Dashboard de la data limpia
Dashboard interactivo construido sobre `df_clean` para explorar métricas clave de estudiantes.

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Base del dashboard
dashboard_df = df_clean.copy()

# KPIs
total_estudiantes = len(dashboard_df)
promedio_nota = dashboard_df["nota"].mean()
promedio_edad = dashboard_df["edad"].mean()
semestre_mas_frecuente = int(dashboard_df["semestre"].mode().iloc[0])

print(f"Total de estudiantes: {total_estudiantes}")
print(f"Promedio de nota: {promedio_nota:.2f}")
print(f"Promedio de edad: {promedio_edad:.1f}")
print(f"Semestre más frecuente: {semestre_mas_frecuente}")

# 1) Distribución de notas
fig_notas = px.histogram(
    dashboard_df,
    x="nota",
    nbins=15,
    title="Distribución de notas",
    labels={"nota": "Nota", "count": "Cantidad de estudiantes"}
 )
fig_notas.update_layout(bargap=0.08)
fig_notas.show()

# 2) Promedio de nota por semestre
nota_por_semestre = (
    dashboard_df.groupby("semestre", as_index=False)["nota"]
    .mean()
    .sort_values("semestre")
)

fig_semestre = px.bar(
    nota_por_semestre,
    x="semestre",
    y="nota",
    title="Promedio de nota por semestre",
    labels={"semestre": "Semestre", "nota": "Nota promedio"},
    text_auto=".2f"
 )
fig_semestre.update_traces(textposition="outside")
fig_semestre.update_layout(yaxis_range=[1, 7.2])
fig_semestre.show()

# 3) Relación edad vs nota
fig_dispersion = px.scatter(
    dashboard_df,
    x="edad",
    y="nota",
    trendline="ols",
    title="Relación entre edad y nota",
    labels={"edad": "Edad", "nota": "Nota"}
 )
fig_dispersion.update_layout(yaxis_range=[1, 7.2])
fig_dispersion.show()

# 4) Composición por semestre (donut)
conteo_semestres = (
    dashboard_df["semestre"]
    .value_counts()
    .sort_index()
    .reset_index()
)
conteo_semestres.columns = ["semestre", "cantidad"]

fig_donut = px.pie(
    conteo_semestres,
    names="semestre",
    values="cantidad",
    hole=0.45,
    title="Proporción de estudiantes por semestre"
 )
fig_donut.show()